<a href="https://colab.research.google.com/github/ranygo12-pixel/AWS/blob/main/Bedrock07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AWS CLI 설치 및 설정

In [ ]:
%%bash

if command -v aws &> /dev/null
then
    echo "AWS CLI is already installed. Updating..."
    curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
    unzip -o awscliv2.zip
    sudo ./aws/install --update
else
    echo "AWS CLI not found. Installing..."
    curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
    unzip -o awscliv2.zip
    sudo ./aws/install
fi

# Clean up the downloaded files
rm awscliv2.zip
rm -rf aws

Lambda 함수 실행 환경 준비

In [ ]:
# Colab에서 Lambda 배포 패키지 생성
!pip install boto3 pygithub

Secrets 설정

In [ ]:
import os
import boto3
from google.colab import userdata

# 1. Colab Secrets에서 자격 증명 불러오기 및 환경 변수 설정
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'ap-northeast-2' # 서울 리전 등 인프라가 구성된 리전 입력
kb_id = userdata.get('KB_ID')

# GitHub Personal Access Token 생성
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')
# print(GITHUB_TOKEN)
agent_id = userdata.get('AGENT_ID')
alias_id = userdata.get('AGENT_ALIAS_ID')

# 2. boto3 클라이언트 생성 (환경 변수를 자동 참조함)
s3_client = boto3.client('s3')

# 3. 연동 테스트: S3 버킷 리스트 출력
try:
    response = s3_client.list_buckets()
    print("✅ AWS 연동 성공! S3 버킷 목록:")
    for bucket in response.get('Buckets', []):
        print(f" - {bucket['Name']}")
except Exception as e:
    print(f"❌ 연결 실패: {e}")

Lambda 실행 IAM 역할 생성

In [ ]:
import boto3
import json
import botocore.exceptions

iam = boto3.client('iam')
sts_client = boto3.client('sts') # Initialize STS client to get account ID

# Lambda용 IAM 역할 생성
lambda_role_name = "CodeBuddy-Lambda-Role"
lambda_function_name = "codebuddy-github-pr"

try:
    lambda_role = iam.create_role(
        RoleName=lambda_role_name,
        AssumeRolePolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Principal": {"Service": "lambda.amazonaws.com"},
                "Action": "sts:AssumeRole"
            }]
        })
    )

    # CloudWatch Logs 권한 부착
    iam.attach_role_policy(
        RoleName=lambda_role_name,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
    )
    lambda_role_arn = lambda_role['Role']['Arn']
    print(f"✅ Lambda 역할 생성 완료: {lambda_role_arn}")
except botocore.exceptions.ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        account_id = sts_client.get_caller_identity()['Account']
        lambda_role_arn = f"arn:aws:iam::{account_id}:role/{lambda_role_name}"
        print(f"⚠️ 기존 역할 사용: {lambda_role_arn}")
    else:
        raise e

GitHub PR 조회 Lambda 함수

In [ ]:
import os
import zipfile
import io
import shutil
import subprocess

def create_deployment_package():
    # 임시 디렉토리 생성
    temp_dir = "./lambda_package"
    os.makedirs(temp_dir, exist_ok=True)

    # Lambda 함수 코드 작성
    # IiOBRQx0RUZ9 셀의 lambda_code 변수를 참조합니다.
    # 이 변수는 해당 셀이 실행되어야 정의됩니다.

    # Lambda 함수 코드 (PyGithub로 PR 정보 조회)
    lambda_code = '''
import json
import os
from github import Github

def handler(event, context):
    # Bedrock Agent가 전달한 파라미터 추출
    parameters = {p["name"]: p["value"] for p in event.get("parameters", [])}

    owner = parameters.get("owner")
    repo_name = parameters.get("repo")
    pr_number = int(parameters.get("pr_number"))

    # GitHub API 호출
    g = Github(os.environ['GITHUB_TOKEN'])
    repo = g.get_repo(f"{owner}/{repo_name}")
    pr = repo.get_pull(pr_number)

    # 응답 생성
    response_body = {
        "title": pr.title,
        "body": pr.body,
        "state": pr.state,
        "author": pr.user.login,
        "created_at": pr.created_at.isoformat(),
        "changed_files": pr.changed_files,
        "additions": pr.additions,
        "deletions": pr.deletions,
        "diff_url": pr.diff_url
    }

    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event["actionGroup"],
            "apiPath": event["apiPath"],
            "httpMethod": event["httpMethod"],
            "httpStatusCode": 200,
            "responseBody": {
                "application/json": {"body": response_body}
            }
        }
    }
'''

    with open(os.path.join(temp_dir, "index.py"), "w") as f:
        f.write(lambda_code)

    # PyGithub 설치 (임시 디렉토리 내에 설치)
    subprocess.check_call(['pip', 'install', 'PyGithub', '-t', temp_dir])

    # ZIP 파일 생성
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(temp_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, temp_dir)
                zf.write(file_path, arcname)
    zip_buffer.seek(0)

    # 임시 디렉토리 삭제
    shutil.rmtree(temp_dir)

    return zip_buffer.getvalue()

In [ ]:
def create_lambda_function(lambda_role_arn, github_token):
    lambda_client = boto3.client('lambda')

    # Lambda 함수 생성
    response = lambda_client.create_function(
        FunctionName='codebuddy-github-pr',
        Runtime='python3.12',
        Role=lambda_role_arn,
        Handler='index.handler',
        Code={'ZipFile': create_deployment_package()},  # 별도 함수
        Environment={'Variables': {'GITHUB_TOKEN': github_token}},
        Timeout=30
    )
    return response['FunctionArn']

In [ ]:
# Lambda 함수 생성
print(f"Lambda 역할 ARN: {lambda_role_arn}")
print(f"GitHub 토큰: {GITHUB_TOKEN}")

try:
    lambda_function_arn = create_lambda_function(lambda_role_arn, GITHUB_TOKEN)
    print(f"✅ Lambda 함수 생성 완료: {lambda_function_arn}")
except Exception as e:
    # 이전에 함수가 생성되었을 경우, ARN을 직접 구성하여 사용
    if "Function already exist" in str(e) or "ResourceConflictException" in str(e):
        account_id = boto3.client('sts').get_caller_identity()['Account']
        lambda_function_arn = f"arn:aws:lambda:ap-northeast-2:{account_id}:function:codebuddy-github-pr"
        print(f"⚠️ 기존 Lambda 함수 사용: {lambda_function_arn}")
    else:
        raise e


Agent → Lambda 호출을 위한 리소스 기반 정책

In [ ]:
lambda_client = boto3.client('lambda')

# Lambda에 Bedrock Agent 호출 권한 추가
try:
    lambda_client.add_permission(
        FunctionName=lambda_function_name,
        StatementId='AllowBedrockAgentInvoke',
        Action='lambda:InvokeFunction',
        Principal='bedrock.amazonaws.com',
        SourceArn=f'arn:aws:bedrock:{os.environ["AWS_DEFAULT_REGION"]}:{account_id}:agent/{agent_id}'
    )
    print("✅ Lambda 함수에 Bedrock Agent 호출 권한 추가 완료")
except botocore.exceptions.ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        print("⚠️ Lambda 함수에 이미 Bedrock Agent 호출 권한이 추가되어 있습니다.")
    else:
        raise e


### Lambda 함수 테스트

In [ ]:
lambda_client = boto3.client('lambda')

# 테스트를 위한 파라미터 설정
# 여기에 실제 GitHub 리포지토리 정보와 PR 번호를 입력하세요.
owner = "octocat"
repo = "Spoon-Knife"
pr_number = 1

# Lambda 함수에 전달할 이벤트 페이로드 구성
# Bedrock Agent 호출 형식을 모방
event_payload = {
    "messageVersion": "1.0",
    "actionGroup": "GithubPRViewer",
    "apiPath": "/github-pr",
    "httpMethod": "GET",
    "parameters": [
        {"name": "owner", "value": owner},
        {"name": "repo", "value": repo},
        {"name": "pr_number", "value": str(pr_number)}
    ]
}

try:
    # Lambda 함수 호출
    response = lambda_client.invoke(
        FunctionName=lambda_function_name,
        InvocationType='RequestResponse', # 동기 호출
        Payload=json.dumps(event_payload)
    )

    # 응답 처리
    response_payload = json.loads(response['Payload'].read().decode('utf-8'))

    print("✅ Lambda 함수 호출 성공!")
    print("--- 응답 본문 ---")
    # responseBody 안에 있는 실제 데이터를 추출하여 보기 좋게 출력
    if 'response' in response_payload and 'responseBody' in response_payload['response']:
        actual_response_body = response_payload['response']['responseBody']['application/json']['body']
        print(json.dumps(actual_response_body, indent=2, ensure_ascii=False))
    else:
        print(json.dumps(response_payload, indent=2, ensure_ascii=False))

except Exception as e:
    print(f"❌ Lambda 함수 호출 실패: {e}")


OpenAPI 스키마 작성

In [ ]:
import json

# get_github_pr 도구의 OpenAPI 스키마
github_pr_tool_openapi_schema = {
  "openapi": "3.0.0",
  "info": {
    "title": "GitHub PR Tools",
    "version": "1.0.0"
  },
  "paths": {
    "/pr": {
      "get": {
        "operationId": "get_github_pr",
        "summary": "Get GitHub Pull Request details",
        "description": "Retrieves detailed information about a specific GitHub Pull Request. Use this when the user asks about PR details, wants to review a PR, or needs to analyze code changes in a pull request.",
        "parameters": [
          {
            "name": "owner",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "The GitHub repository owner/organization name (e.g., 'aws-samples', 'facebook')"
          },
          {
            "name": "repo",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "The GitHub repository name (e.g., 'amazon-bedrock-samples')"
          },
          {
            "name": "pr_number",
            "in": "query",
            "required": True,
            "schema": {"type": "integer"},
            "description": "The Pull Request number to retrieve"
          }
        ],
        "responses": {
          "200": {
            "description": "PR details retrieved successfully",
            "content": {"application/json": {"schema": {"type": "object"}}}
          }
        }
      }
    }
  }
}

## Boto3로 Action Group 생성
create_agent_action_group() API로 Action Group 생성

In [ ]:
def create_action_group(agent_id, lambda_arn, api_schema):
    bedrock_agent = boto3.client('bedrock-agent')

    response = bedrock_agent.create_agent_action_group(
        agentId=agent_id,
        agentVersion="DRAFT",
        actionGroupName="GitHubPRTools",
        actionGroupExecutor={
            "lambda": lambda_arn  # Lambda 함수 ARN
        },
        apiSchema={
            "payload": json.dumps(api_schema)  # OpenAPI 스키마
        },
        actionGroupState="ENABLED",
        description="Tools for fetching GitHub Pull Request information"
    )

    action_group_id = response['agentActionGroup']['actionGroupId']
    print(f"✅ Action Group 생성 완료!")
    print(f"   Action Group ID: {action_group_id}")

    return action_group_id

Action Group 추가 후 Agent Prepare

In [ ]:
# Agent Prepare 재실행
def prepare_agent_with_retry(agent_id, max_wait=60):
    bedrock_agent = boto3.client('bedrock-agent')

    # Prepare 실행
    bedrock_agent.prepare_agent(agentId=agent_id)
    print("⏳ Agent Prepare 중... (최대 60초)")

    # Prepare 완료 대기
    for i in range(max_wait):
        agent_info = bedrock_agent.get_agent(agentId=agent_id)
        status = agent_info['agent']['agentStatus']

        if status == 'PREPARED':
            print(f"✅ Agent Prepare 완료! (소요 시간: {i+1}초)")
            return True
        elif status == 'FAILED':
            print(f"❌ Prepare 실패!")
            return False
        time.sleep(1)

    print("⏰ Prepare 시간 초과")
    return False

# Agent Alias도 새로 생성 (또는 업데이트)
def update_alias(agent_id, alias_id):
    bedrock_agent = boto3.client('bedrock-agent')

    return bedrock_agent.update_agent_alias(
        agentId=agent_id,
        agentAliasId=alias_id,
        agentAliasName="dev",
        description="Updated with GitHub PR tool"
    )

Agent에 Tool 사용 지침 추가

In [ ]:
# Agent 업데이트를 위한 Bedrock Agent 클라이언트 초기화
import boto3

bedrock_agent = boto3.client('bedrock-agent')

# 현재 Agent의 Instructions 가져오기
agent_info = bedrock_agent.get_agent(agentId=agent_id)['agent']
agent_instruction = agent_info['instruction']

# Instructions에 Tool 사용법 추가
updated_instruction = agent_instruction + """

## 사용 가능한 도구 (Tools)
- get_github_pr: GitHub Pull Request 정보를 가져옵니다.
  * 사용 예시: "my-org/my-repo의 PR #123 가져와줘"
  * 파라미터: owner (저장소 소유자), repo (저장소 이름), pr_number (PR 번호)
  * 이 도구는 사용자가 PR 정보를 요청할 때 사용하세요.

## 도구 사용 규칙
1. 사용자가 PR 정보를 요청하면 get_github_pr 도구를 호출하세요.
2. 필수 파라미터(owner, repo, pr_number)가 없는 경우 사용자에게 물어보세요.
3. 도구 호출 결과를 사용자에게 이해하기 쉽게 설명해주세요.
"""

# Agent 업데이트
bedrock_agent.update_agent(
    agentId=agent_id,
    agentName=agent_info['agentName'],
    agentResourceRoleArn=agent_info['agentResourceRoleArn'],
    foundationModel=agent_info['foundationModel'],
    instruction=updated_instruction
)

In [ ]:
# Agent 업데이트를 위한 Bedrock Agent 클라이언트 초기화
import boto3

bedrock_agent = boto3.client('bedrock-agent')

# 현재 Agent의 Instructions 가져오기
agent_info = bedrock_agent.get_agent(agentId=agent_id)['agent']

# Remove duplicate instructions, if any, and ensure unique structure
current_instruction = agent_info['instruction']

# Split existing instructions by the known header and re-assemble to avoid duplication
instruction_parts = current_instruction.split("## 사용 가능한 도구 (Tools)")
base_instruction = instruction_parts[0].strip()

# Define new instructions for both tools
new_tools_instruction = """

## 사용 가능한 도구 (Tools)
- get_github_pr: GitHub Pull Request 정보를 가져옵니다.
  * 사용 예시: "my-org/my-repo의 PR #123 가져와줘"
  * 파라미터: owner (저장소 소유자), repo (저장소 이름), pr_number (PR 번호)
  * 이 도구는 사용자가 PR 정보를 요청할 때 사용하세요.

- post_pr_comment: GitHub Pull Request에 댓글을 추가합니다.
  * 사용 예시: "my-org/my-repo의 PR #123에 '잘했어요! LGTM' 댓글 남겨줘"
  * 파라미터: owner (저장소 소유자), repo (저장소 이름), pr_number (PR 번호), comment (댓글 내용)
  * 이 도구는 사용자가 PR에 피드백, 승인, 질문 등의 댓글을 남기고자 할 때 사용하세요.

## 도구 사용 규칙
1. 사용자가 PR 정보 조회 또는 댓글 요청을 하면 해당 도구를 호출하세요.
2. 필수 파라미터(owner, repo, pr_number, comment)가 없는 경우 사용자에게 물어보세요.
3. 도구 호출 결과를 사용자에게 이해하기 쉽게 설명해주세요.
"""

# Combine base instruction with new tool instructions
updated_instruction = base_instruction + new_tools_instruction

# Agent 업데이트
bedrock_agent.update_agent(
    agentId=agent_id,
    agentName=agent_info['agentName'],
    agentResourceRoleArn=agent_info['agentResourceRoleArn'],
    foundationModel=agent_info['foundationModel'],
    instruction=updated_instruction
)

print("✅ Agent Instructions updated with both `get_github_pr` and `post_pr_comment` tools.")

Agent 호출로 Tool 실행 테스트

In [ ]:
def test_github_pr_tool(agent_id, alias_id):
    bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')

    # PR 정보 요청 (실제 존재하는 공개 리포지토리와 PR 번호로 변경)
    # 'octocat/Spoon-Knife'는 잘 알려진 공개 리포지토리이며, #540은 현재 유효한 Pull Request입니다.
    prompt = "octocat/Spoon-Knife 저장소의 Pull Request #540 정보를 가져와줘"

    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId="test-session-001",
        inputText=prompt,
        enableTrace=True
    )

    # 응답 처리
    print("🤖 Agent 응답:")
    for event in response['completion']:
        if 'chunk' in event:
            print(event['chunk']['bytes'].decode('utf-8'), end='')
        elif 'trace' in event:
            # 도구 호출 과정 확인
            trace_info = event['trace']['trace']
            if 'orchestrationTrace' in trace_info:
                orchestration_trace_data = trace_info['orchestrationTrace']
                if 'invocationInput' in orchestration_trace_data:
                    print(f"\n🔧 Tool 호출: {orchestration_trace_data['invocationInput']}")
                elif 'observation' in orchestration_trace_data:
                    print(f"\n💡 Agent Observation: {orchestration_trace_data['observation']}")
                elif 'modelThought' in orchestration_trace_data:
                    print(f"\n🤔 Agent Thought: {orchestration_trace_data['modelThought']}")

# 실행
test_github_pr_tool(agent_id, alias_id)

Tool 호출 로그 분석

In [ ]:
logs = boto3.client('logs')

def get_lambda_logs(lambda_name, hours=1):
    """Lambda 함수의 최근 로그 확인"""

    # 로그 그룹 확인
    log_group = f'/aws/lambda/{lambda_name}'

    # 최근 로그 스트림 찾기
    streams = logs.describe_log_streams(
        logGroupName=log_group,
        orderBy='LastEventTime',
        descending=True,
        limit=1
    )

    if not streams['logStreams']:
        return "로그 없음"

    # 로그 이벤트 가져오기
    events = logs.get_log_events(
        logGroupName=log_group,
        logStreamName=streams['logStreams'][0]['logStreamName']
    )

    for event in events['events']:
        print(f"[{event['timestamp']}] {event['message']}")

# 실행 (Lambda 호출 후)
get_lambda_logs('codebuddy-github-pr')

미션: Repository 목록 조회 Tool 추가

In [ ]:
import json

github_list_repos_tool_openapi_schema = {
  "openapi": "3.0.0",
  "info": {
    "title": "GitHub Repository Tools",
    "version": "1.0.0"
  },
  "paths": {
    "/repos": {
      "get": {
        "operationId": "list_repositories",
        "summary": "List user's GitHub repositories",
        "description": "Lists all repositories for a GitHub user/organization. Use this when the user wants to see their repositories or find a specific repo.",
        "parameters": [
          {
            "name": "owner",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "GitHub username or organization name"
          },
          {
            "name": "type",
            "in": "query",
            "required": False,
            "schema": {"type": "string", "enum": ["all", "public", "private"]},
            "description": "Filter by repository type (all, public, private)"
          }
        ],
        "responses": {
          "200": {
            "description": "Repository list retrieved successfully",
            "content": {"application/json": {"schema": {"type": "array", "items": {"type": "object"}}}}
          }
        }
      }
    }
  }
}

In [ ]:
LAMBDA_LIST_REPOS_FULL_CODE = '''
import json
import os
import logging
from github import Github, GithubException

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    logger.info(f"Received event for list_repositories: {json.dumps(event)}")

    try:
        parameters = {p["name"]: p["value"] for p in event.get("parameters", [])}

        owner = parameters.get("owner")
        repo_type = parameters.get("type", "all") # Default to 'all' if not provided

        if not owner:
            raise ValueError("Missing required parameter: owner")

        g = Github(os.environ['GITHUB_TOKEN'])

        # Get user or organization to list repositories
        try:
            user = g.get_user(owner)
        except GithubException:
            try:
                user = g.get_organization(owner)
            except GithubException as e:
                logger.error(f"GitHub user or organization not found for '{owner}': {e}")
                return build_error_response(event, f"GitHub user or organization not found for '{owner}'", 404)

        repos = []
        for repo in user.get_repos(type=repo_type):
            repos.append({
                "name": repo.name,
                "full_name": repo.full_name,
                "description": repo.description,
                "html_url": repo.html_url,
                "private": repo.private,
                "fork": repo.fork,
                "created_at": repo.created_at.isoformat(),
                "updated_at": repo.updated_at.isoformat(),
                "language": repo.language
            })

        return {
            "messageVersion": "1.0",
            "response": {
                "actionGroup": event.get("actionGroup"),
                "apiPath": event.get("apiPath"),
                "httpMethod": event.get("httpMethod"),
                "httpStatusCode": 200,
                "responseBody": {
                    "application/json": {"body": repos}
                }
            }
        }

    except GithubException as e:
        logger.error(f"GitHub API error in list_repositories: {e}")
        return build_error_response(event, f"GitHub API error: {e.data.get('message', str(e))}", 404)

    except Exception as e:
        logger.error(f"Unexpected error in list_repositories: {e}")
        return build_error_response(event, str(e), 500)

def build_error_response(event, error_msg, status_code):
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event.get("actionGroup"),
            "apiPath": event.get("apiPath"),
            "httpMethod": event.get("httpMethod"),
            "httpStatusCode": status_code,
            "responseBody": {
                "application/json": {"body": {"error": error_msg}}
            }
        }
    }
'''

In [ ]:
import os
import zipfile
import io
import shutil
import subprocess

def create_list_repos_deployment_package():
    # 임시 디렉토리 생성
    temp_dir = "./lambda_list_repos_package"
    os.makedirs(temp_dir, exist_ok=True)

    # LAMBDA_LIST_REPOS_FULL_CODE 변수를 사용합니다.
    global LAMBDA_LIST_REPOS_FULL_CODE

    with open(os.path.join(temp_dir, "index.py"), "w") as f:
        f.write(LAMBDA_LIST_REPOS_FULL_CODE)

    # PyGithub 설치 (임시 디렉토리 내에 설치)
    subprocess.check_call(['pip', 'install', 'PyGithub', '-t', temp_dir])

    # ZIP 파일 생성
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(temp_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, temp_dir)
                zf.write(file_path, arcname)
    zip_buffer.seek(0)

    # 임시 디렉토리 삭제
    shutil.rmtree(temp_dir)

    return zip_buffer.getvalue()

In [ ]:
list_repos_deployment_package = create_list_repos_deployment_package()
print("✅ 'list_repositories' Lambda deployment package created.")

In [ ]:
# (This cell has been modified previously to update existing Lambda functions)
# GitHub PR Lambda 함수 생성 또는 업데이트
print(f"Lambda 역할 ARN: {lambda_role_arn}")
print(f"GitHub 토큰: {GITHUB_TOKEN}")

# create_deployment_package()는 가장 최근 정의된 것을 사용합니다 (4231ec02 셀 참조)
pr_lambda_deployment_package = create_deployment_package()

try:
    lambda_function_arn = create_lambda_function(lambda_role_arn, lambda_function_name, GITHUB_TOKEN, pr_lambda_deployment_package)
    print(f"✅ Lambda 함수 처리 완료: {lambda_function_arn}")
except Exception as e:
    # 오류 발생 시 ARN을 직접 구성하여 사용
    account_id = boto3.client('sts').get_caller_identity()['Account'] # account_id가 정의되지 않았을 경우를 대비
    lambda_function_arn = f"arn:aws:lambda:ap-northeast-2:{account_id}:function:{lambda_function_name}"
    print(f"❌ Lambda 함수 처리 실패 또는 오류 발생: {e}")
    print(f"⚠️ 기존 Lambda 함수 ARN 사용 시도: {lambda_function_arn}")

Lambda 오류 핸들링 구현

In [ ]:
# 완성된 Lambda 함수 코드 전체
LAMBDA_FULL_CODE = '''
import json
import os
import logging
from github import Github, GithubException

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    logger.info(f"Received event: {json.dumps(event)}")

    try:
        # 파라미터 추출
        parameters = {p["name"]: p["value"] for p in event.get("parameters", [])}

        owner = parameters.get("owner")
        repo_name = parameters.get("repo")
        pr_number = parameters.get("pr_number")

        # 필수 파라미터 검증
        if not all([owner, repo_name, pr_number]):
            raise ValueError(f"Missing required parameters. Need owner, repo, pr_number")

        # GitHub API 호출
        g = Github(os.environ['GITHUB_TOKEN'])
        repo = g.get_repo(f"{owner}/{repo_name}")
        pr = repo.get_pull(int(pr_number))

        # 성공 응답 생성
        response_body = {
            "title": pr.title,
            "body": pr.body or "",
            "state": pr.state,
            "author": pr.user.login,
            "created_at": pr.created_at.isoformat(),
            "updated_at": pr.updated_at.isoformat(),
            "changed_files": pr.changed_files,
            "additions": pr.additions,
            "deletions": pr.deletions,
            "diff_url": pr.diff_url
        }

        return {
            "messageVersion": "1.0",
            "response": {
                "actionGroup": event.get("actionGroup"),
                "apiPath": event.get("apiPath"),
                "httpMethod": event.get("httpMethod"),
                "httpStatusCode": 200,
                "responseBody": {
                    "application/json": {"body": response_body}
                }
            }
        }

    except GithubException as e:
        logger.error(f"GitHub API error: {e}")
        return build_error_response(event, f"GitHub API error: {e.data.get('message', str(e))}", 404)

    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        return build_error_response(event, str(e), 500)

def build_error_response(event, error_msg, status_code):
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event.get("actionGroup"),
            "apiPath": event.get("apiPath"),
            "httpMethod": event.get("httpMethod"),
            "httpStatusCode": status_code,
            "responseBody": {
                "application/json": {"body": {"error": error_msg}}
            }
        }
    }
'''

In [ ]:
import boto3
import time # Import time for waiting

def update_lambda_function(lambda_function_name, lambda_role_arn, github_token, deployment_package):
    lambda_client = boto3.client('lambda')

    # Lambda 함수의 코드 업데이트
    lambda_client.update_function_code(
        FunctionName=lambda_function_name,
        ZipFile=deployment_package
    )
    print(f"⌛ Lambda 함수 '{lambda_function_name}' 코드 업데이트 중...")
    # Wait until the function is updated and active
    waiter = lambda_client.get_waiter('function_updated_v2')
    waiter.wait(FunctionName=lambda_function_name, WaiterConfig={'Delay': 2, 'MaxAttempts': 30}) # Wait up to 60 seconds
    print(f"✅ Lambda 함수 '{lambda_function_name}' 코드 업데이트 완료.")

    # Lambda 함수의 환경 변수 및 역할 업데이트
    lambda_client.update_function_configuration(
        FunctionName=lambda_function_name,
        Role=lambda_role_arn,
        Environment={'Variables': {'GITHUB_TOKEN': github_token}}
    )

    print(f"✅ Lambda 함수 '{lambda_function_name}' 설정 업데이트 완료.")

    # 업데이트된 함수 ARN 반환 (여기서는 기존 ARN을 사용)
    account_id = boto3.client('sts').get_caller_identity()['Account']
    return f"arn:aws:lambda:ap-northeast-2:{account_id}:function:{lambda_function_name}"

Now, let's call the updated helper function to deploy the changes to our `codebuddy-github-pr` Lambda function.

In [ ]:
# Ensure the latest deployment package is created using the corrected LAMBDA_FULL_CODE
pr_lambda_deployment_package = create_deployment_package()

# Call the new update function
updated_lambda_function_arn = update_lambda_function(
    lambda_function_name, # from previous cells
    lambda_role_arn,      # from previous cells
    GITHUB_TOKEN,         # from previous cells
    pr_lambda_deployment_package
)

print(f"Updated Lambda Function ARN: {updated_lambda_function_arn}")

After updating the Lambda function, we should re-run the test to ensure the syntax error is resolved.

In [ ]:
# Re-run the test for the GitHub PR tool
def test_github_pr_tool(agent_id, alias_id):
    bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')

    # PR 정보 요청 (실제 존재하는 공개 리포지토리와 PR 번호로 변경)
    # 'octocat/Spoon-Knife'는 잘 알려진 공개 리포지토리이며, #540은 현재 유효한 Pull Request입니다.
    prompt = "octocat/Spoon-Knife 저장소의 Pull Request #540 정보를 가져와줘"

    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId="test-session-001",
        inputText=prompt,
        enableTrace=True
    )

    # 응답 처리
    print("🤖 Agent 응답:")
    for event in response['completion']:
        if 'chunk' in event:
            print(event['chunk']['bytes'].decode('utf-8'), end='')
        elif 'trace' in event:
            # 도구 호출 과정 확인
            trace_info = event['trace']['trace']
            if 'orchestrationTrace' in trace_info:
                # Correctly handle orch variable to avoid UnboundLocalError
                orchestration_trace_data = trace_info['orchestrationTrace']
                if 'invocationInput' in orchestration_trace_data:
                    print(f"\n🔧 Tool 호출: {orchestration_trace_data['invocationInput']}")
                # If there's no invocationInput, it means the agent might be planning or responding directly.
                # We can add more detailed logging here if needed.
                elif 'observation' in orchestration_trace_data:
                    print(f"\n💡 Agent Observation: {orchestration_trace_data['observation']}")
                elif 'modelThought' in orchestration_trace_data:
                    print(f"\n🤔 Agent Thought: {orchestration_trace_data['modelThought']}")

# 실행
test_github_pr_tool(agent_id, alias_id)

In [ ]:
import os
import zipfile
import io
import shutil
import subprocess

def create_deployment_package():
    # 임시 디렉토리 생성
    temp_dir = "./lambda_package"
    os.makedirs(temp_dir, exist_ok=True)

    # ALppeiMaS3ID 셀에서 정의된 LAMBDA_FULL_CODE 변수를 사용합니다.
    # 이 변수는 해당 셀이 먼저 실행되어야 합니다.
    global LAMBDA_FULL_CODE

    with open(os.path.join(temp_dir, "index.py"), "w") as f:
        f.write(LAMBDA_FULL_CODE)

    # PyGithub 설치 (임시 디렉토리 내에 설치)
    # 필요한 경우 PyGithub 버전을 지정할 수 있습니다.
    subprocess.check_call(['pip', 'install', 'PyGithub', '-t', temp_dir])

    # ZIP 파일 생성
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(temp_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, temp_dir)
                zf.write(file_path, arcname)
    zip_buffer.seek(0)

    # 임시 디렉토리 삭제
    shutil.rmtree(temp_dir)

    return zip_buffer.getvalue()

## 두 번째 Tool: PR에 댓글 남기기(7장)
Lambda 함수 (PR 댓글): PR 댓글 Lambda 함수 구현

In [ ]:
# lambda_post_comment.py
import json
import os
import logging
from github import Github, GithubException

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    logger.info(f"Received event: {json.dumps(event)}")

    try:
        # 파라미터 추출
        parameters = {p["name"]: p["value"] for p in event.get("parameters", [])}

        owner = parameters.get("owner")
        repo_name = parameters.get("repo")
        pr_number = int(parameters.get("pr_number"))
        comment = parameters.get("comment")

        # 필수 파라미터 검증
        if not all([owner, repo_name, pr_number, comment]):
            raise ValueError("Missing required parameters")

        # GitHub API 호출
        g = Github(os.environ['GITHUB_TOKEN'])
        repo = g.get_repo(f"{owner}/{repo_name}")
        pr = repo.get_pull(pr_number)
        pr.create_issue_comment(comment)

        response_body = {
            "success": True,
            "message": f"Comment added to PR #{pr_number}",
            "pr_url": pr.html_url
        }

        return build_success_response(event, response_body)

    except GithubException as e:
        logger.error(f"GitHub API error: {e}")
        return build_error_response(event, str(e), 400)
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        return build_error_response(event, str(e), 500)

In [ ]:
LAMBDA_POST_COMMENT_FULL_CODE = '''
import json
import os
import logging
from github import Github, GithubException

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    logger.info(f"Received event: {json.dumps(event)}")

    try:
        # 파라미터 추출
        parameters = {p["name"]: p["value"] for p in event.get("parameters", [])}

        owner = parameters.get("owner")
        repo_name = parameters.get("repo")
        pr_number = int(parameters.get("pr_number"))
        comment = parameters.get("comment")

        # 필수 파라미터 검증
        if not all([owner, repo_name, pr_number, comment]):
            raise ValueError("Missing required parameters")

        # GitHub API 호출
        g = Github(os.environ['GITHUB_TOKEN'])
        repo = g.get_repo(f"{owner}/{repo_name}")
        pr = repo.get_pull(pr_number)
        pr.create_issue_comment(comment)

        response_body = {
            "success": True,
            "message": f"Comment added to PR #{pr_number}",
            "pr_url": pr.html_url
        }

        return build_success_response(event, response_body)

    except GithubException as e:
        logger.error(f"GitHub API error: {e}")
        return build_error_response(event, str(e), 400)
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        return build_error_response(event, str(e), 500)

def build_error_response(event, error_msg, status_code):
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event.get("actionGroup"),
            "apiPath": event.get("apiPath"),
            "httpMethod": event.get("httpMethod"),
            "httpStatusCode": status_code,
            "responseBody": {
                "application/json": {"body": {"error": error_msg}}
            }
        }
    }

def build_success_response(event, response_body):
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event.get("actionGroup"),
            "apiPath": event.get("apiPath"),
            "httpMethod": event.get("httpMethod"),
            "httpStatusCode": 200,
            "responseBody": {
                "application/json": {"body": response_body}
            }
        }
    }
'''

print("✅ `LAMBDA_POST_COMMENT_FULL_CODE` defined.")

In [ ]:
def create_post_comment_deployment_package():
    # 임시 디렉토리 생성
    temp_dir = "./lambda_post_comment_package"
    os.makedirs(temp_dir, exist_ok=True)

    global LAMBDA_POST_COMMENT_FULL_CODE

    with open(os.path.join(temp_dir, "index.py"), "w") as f:
        f.write(LAMBDA_POST_COMMENT_FULL_CODE)

    # PyGithub 설치 (임시 디렉토리 내에 설치)
    subprocess.check_call(['pip', 'install', 'PyGithub', '-t', temp_dir])

    # ZIP 파일 생성
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(temp_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, temp_dir)
                zf.write(file_path, arcname)
    zip_buffer.seek(0)

    # 임시 디렉토리 삭제
    shutil.rmtree(temp_dir)

    return zip_buffer.getvalue()

post_comment_deployment_package = create_post_comment_deployment_package()
print("✅ 'post_pr_comment' Lambda deployment package created.")

### 새로운 Lambda 함수 배포: `codebuddy-github-pr-comment`

In [ ]:
post_comment_lambda_function_name = "codebuddy-github-pr-comment"

try:
    # Try to create, if exists, update
    lambda_client = boto3.client('lambda')
    lambda_client.create_function(
        FunctionName=post_comment_lambda_function_name,
        Runtime='python3.12',
        Role=lambda_role_arn,
        Handler='index.handler',
        Code={'ZipFile': post_comment_deployment_package},
        Environment={'Variables': {'GITHUB_TOKEN': GITHUB_TOKEN}},
        Timeout=30
    )
    post_comment_lambda_function_arn = f"arn:aws:lambda:ap-northeast-2:{account_id}:function:{post_comment_lambda_function_name}"
    print(f"✅ Lambda 함수 생성 완료: {post_comment_lambda_function_arn}")
except Exception as e:
    if "ResourceConflictException" in str(e):
        post_comment_lambda_function_arn = update_lambda_function(
            post_comment_lambda_function_name,
            lambda_role_arn,
            GITHUB_TOKEN,
            post_comment_deployment_package
        )
        print(f"⚠️ 기존 Lambda 함수 업데이트 완료: {post_comment_lambda_function_arn}")
    else:
        raise e

# Add permissions for Bedrock Agent to invoke this new Lambda function
try:
    lambda_client.add_permission(
        FunctionName=post_comment_lambda_function_name,
        StatementId='AllowBedrockAgentInvokeForComment',
        Action='lambda:InvokeFunction',
        Principal='bedrock.amazonaws.com',
        SourceArn=f'arn:aws:bedrock:{os.environ["AWS_DEFAULT_REGION"]}:{account_id}:agent/{agent_id}'
    )
    print("✅ `codebuddy-github-pr-comment` 함수에 Bedrock Agent 호출 권한 추가 완료")
except botocore.exceptions.ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        print("⚠️ `codebuddy-github-pr-comment` 함수에 이미 Bedrock Agent 호출 권한이 추가되어 있습니다.")
    else:
        raise e

## GitHub API 호출 상세
GitHub API 호출 상세 (POST /issues/comments)

POST /repos/{owner}/{repo}/issues/{issue_number}/comments

직접 Requests 방식 예시

In [ ]:
import requests

def add_pr_comment(owner, repo, pr_number, comment, token):
    url = f"https://api.github.com/repos/{owner}/{repo}/issues/{pr_number}/comments"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github.v3+json"
    }
    response = requests.post(url, headers=headers, json={"body": comment})
    response.raise_for_status()
    return response.json()

## OpenAPI 스키마에 post_pr_comment 추가
기존 OpenAPI 스키마에 새 경로 추가

In [ ]:
{
  "openapi": "3.0.0",
  "info": {
    "title": "GitHub PR Tools",
    "version": "1.0.0"
  },
  "paths": {
    "/pr": {
      "get": {
        "operationId": "get_github_pr",
        "summary": "Get GitHub Pull Request details",
        "description": "Retrieves detailed information about a specific GitHub Pull Request. Use this when the user asks about PR details, wants to review a PR, or needs to analyze code changes in a pull request.",
        "parameters": [
          {
            "name": "owner",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "The GitHub repository owner/organization name (e.g., 'aws-samples', 'facebook')"
          },
          {
            "name": "repo",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "The GitHub repository name (e.g., 'amazon-bedrock-samples')"
          },
          {
            "name": "pr_number",
            "in": "query",
            "required": True,
            "schema": {"type": "integer"},
            "description": "The Pull Request number to retrieve"
          }
        ],
        "responses": {
          "200": {
            "description": "PR details retrieved successfully",
            "content": {"application/json": {"schema": {"type": "object"}}}
          }
        }
      }
    },
    "/pr/comment": {
      "post": {
        "operationId": "post_pr_comment",
        "summary": "Add a comment to a Pull Request",
        "description": "Adds a comment to a specific GitHub Pull Request. Use this when the user wants to leave feedback, approval, or questions on a PR.",
        "parameters": [
          {
            "name": "owner",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "Repository owner/organization"
          },
          {
            "name": "repo",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "Repository name"
          },
          {
            "name": "pr_number",
            "in": "query",
            "required": True,
            "schema": {"type": "integer"},
            "description": "Pull Request number"
          },
          {
            "name": "comment",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "The comment text to post"
          }
        ],
        "responses": {
          "200": {"description": "Comment added successfully"}
        }
      }
    }
  }
}

# 두 번째 Tool을 기존 Action Group에 추가
Action Group 업데이트 (Tool 추가)

In [ ]:
import boto3
import json
import time # Import time for retries

# --- Helper function for finding existing action group ---
def find_action_group_id(agent_id, action_group_name, max_retries=15, retry_delay_seconds=10):
    bedrock_agent_client = boto3.client('bedrock-agent')
    for attempt in range(max_retries):
        print(f"Attempt {attempt + 1}/{max_retries}: Listing action groups for agent {agent_id} to find '{action_group_name}'...")
        list_response = bedrock_agent_client.list_agent_action_groups(
            agentId=agent_id,
            agentVersion="DRAFT"
        )
        summaries = list_response.get('agentActionGroupSummaries', [])
        print(f"  -> Found {len(summaries)} action groups. Summaries: {summaries}") # Diagnostic print

        for ag_summary in summaries:
            if ag_summary['actionGroupName'] == action_group_name:
                print(f"✅ Found existing Action Group '{action_group_name}' with ID: {ag_summary['actionGroupId']}.")
                return ag_summary['actionGroupId']
        print(f"Action Group '{action_group_name}' not found. Retrying in {retry_delay_seconds} seconds...")
        time.sleep(retry_delay_seconds)
    return None

# --- Modified update_action_group_with_new_tool to accept action_group_id directly ---
def update_action_group_with_id(agent_id, action_group_id, lambda_arn, updated_api_schema, action_group_name="GitHubPRTools"):
    bedrock_agent_client = boto3.client('bedrock-agent')

    response = bedrock_agent_client.update_agent_action_group(
        agentId=agent_id,
        agentVersion="DRAFT",
        actionGroupId=action_group_id,
        actionGroupName=action_group_name, # This is optional for update, but harmless
        actionGroupExecutor={
            "lambda": lambda_arn  # Lambda 함수 ARN
        },
        apiSchema={
            "payload": json.dumps(updated_api_schema)  # OpenAPI 스키마
        },
        actionGroupState="ENABLED",
        description="GitHub PR tools: get PR, add comment"
    )

    print("✅ Action Group 업데이트 완료")

    return response

In [ ]:
import boto3
import json
import time

bedrock_agent = boto3.client('bedrock-agent')
action_group_name = "GitHubPRTools"
target_action_group_id = None

# Try to find the action group first
target_action_group_id = find_action_group_id(agent_id, action_group_name)

if not target_action_group_id:
    print(f"Action Group '{action_group_name}' not found after initial search. Attempting to create it.")
    try:
        # Assuming github_pr_tool_openapi_schema and lambda_function_arn are defined from previous cells
        create_response = bedrock_agent.create_agent_action_group(
            agentId=agent_id,
            agentVersion="DRAFT",
            actionGroupName=action_group_name,
            actionGroupExecutor={
                "lambda": lambda_function_arn # The original lambda for get_github_pr
            },
            apiSchema={
                "payload": json.dumps(github_pr_tool_openapi_schema) # The schema for get_github_pr only
            },
            actionGroupState="ENABLED",
            description="Tools for fetching GitHub Pull Request information"
        )
        target_action_group_id = create_response['agentActionGroup']['actionGroupId']
        print(f"✅ Action Group '{action_group_name}' created with ID: {target_action_group_id}.")
    except bedrock_agent.exceptions.ConflictException:
        print(f"⚠️ Action Group '{action_group_name}' already exists (ConflictException during create). Retrying to find its ID.")
        # If it conflicted, it must exist, so retry finding it.
        target_action_group_id = find_action_group_id(agent_id, action_group_name, max_retries=10, retry_delay_seconds=6)
        if not target_action_group_id:
            raise ValueError(f"Failed to find Action Group '{action_group_name}' even after ConflictException.")
    except Exception as e:
        raise e # Re-raise other exceptions

if not target_action_group_id:
    raise ValueError(f"Could not determine Action Group ID for '{action_group_name}'. Exiting.")

# Get the combined OpenAPI schema (from cell k-E6GYwcbmwG)
# combined_openapi_schema is already defined in the kernel state.

# Update the action group with the combined OpenAPI schema using the found/created ID
updated_action_group_response = update_action_group_with_id(
    agent_id,
    target_action_group_id,
    post_comment_lambda_function_arn, # Executor for the combined schema (assumes it handles both tools)
    combined_openapi_schema
)

print("✅ Action Group updated with new PR Comment tool.")


In [ ]:
# Prepare the agent after updating the action group
prepare_agent_with_retry(agent_id)

# Update the agent alias
update_alias(agent_id, alias_id)
print("✅ Agent prepared and alias updated.")

# PR 댓글 Tool 테스트
Agent 호출로 댓글 남기기

In [ ]:
def test_pr_comment(agent_id, alias_id):
    bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')

    prompt = "내 저장소 my-org/my-repo의 PR #123에 '잘했어요! LGTM' 댓글 남겨줘"

    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId="test-comment-session",
        inputText=prompt,
        enableTrace=True
    )

    # 응답 처리
    for event in response['completion']:
        if 'chunk' in event:
            print(event['chunk']['bytes'].decode('utf-8'))
        elif 'trace' in event:
            trace = event['trace']['trace']
            if 'orchestrationTrace' in trace:
                orch = trace['orchestrationTrace']
                if 'invocationInput' in orch:
                    print(f"🔧 Tool 호출: {orch['invocationInput']}")

# 실행
test_pr_comment(agent_id, alias_id)

### `post_pr_comment` 툴 테스트

In [ ]:
def test_pr_comment_new(agent_id, alias_id):
    bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')

    # Replace with a real repo and PR number where you have write access for testing
    # For this example, we'll try with a fictional repo first. If it fails,
    # the agent should indicate that it tried to call the tool.
    prompt = "내 저장소 octocat/Spoon-Knife의 PR #540에 '잘했어요! LGTM' 댓글 남겨줘"

    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId="test-comment-session-new",
        inputText=prompt,
        enableTrace=True
    )

    # 응답 처리
    for event in response['completion']:
        if 'chunk' in event:
            print(event['chunk']['bytes'].decode('utf-8'))
        elif 'trace' in event:
            trace = event['trace']['trace']
            if 'orchestrationTrace' in trace:
                orch = trace['orchestrationTrace']
                if 'invocationInput' in orch:
                    print(f"🔧 Tool 호출: {orch['invocationInput']}")
                elif 'observation' in orch:
                    print(f"💡 Agent Observation: {orch['observation']}")
                elif 'modelThought' in orch:
                    print(f"🤔 Agent Thought: {orch['modelThought']}")

# 실행
test_pr_comment_new(agent_id, alias_id)

## 세 번째 Tool 설계 (Slack 알림)
Slack 메시지 전송 Lambda 함수
Lambda 코드 (send_slack_message)

In [ ]:
# lambda_slack.py
import json
import os
import requests
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    logger.info(f"Event: {json.dumps(event)}")

    try:
        params = {p["name"]: p["value"] for p in event.get("parameters", [])}
        channel = params.get("channel")
        message = params.get("message")

        if not channel or not message:
            raise ValueError("Missing channel or message")

        webhook_url = os.environ['SLACK_WEBHOOK_URL']

        # Slack 메시지 포맷 (선택적: 채널 지정)
        payload = {"text": message}
        if channel:
            payload["channel"] = channel

        response = requests.post(webhook_url, json=payload)
        response.raise_for_status()

        result = {"success": True, "message": "Slack message sent"}
        return build_success_response(event, result)

    except Exception as e:
        logger.error(f"Error: {e}")
        return build_error_response(event, str(e), 500)

In [ ]:
LAMBDA_SLACK_FULL_CODE = '''
import json
import os
import requests
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    logger.info(f"Event: {json.dumps(event)}")

    try:
        params = {p["name"]: p["value"] for p in event.get("parameters", [])}
        channel = params.get("channel")
        message = params.get("message")

        if not channel or not message:
            raise ValueError("Missing channel or message")

        # SLACK_WEBHOOK_URL은 Lambda 환경 변수로 설정됩니다.
        webhook_url = os.environ['SLACK_WEBHOOK_URL']

        # Slack 메시지 포맷 (선택적: 채널 지정)
        payload = {"text": message}
        if channel:
            payload["channel"] = channel

        response = requests.post(webhook_url, json=payload)
        response.raise_for_status() # HTTP 오류 발생 시 예외 발생

        result = {"success": True, "message": "Slack message sent"}
        return build_success_response(event, result)

    except Exception as e:
        logger.error(f"Error: {e}")
        return build_error_response(event, str(e), 500)

def build_error_response(event, error_msg, status_code):
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event.get("actionGroup"),
            "apiPath": event.get("apiPath"),
            "httpMethod": event.get("httpMethod"),
            "httpStatusCode": status_code,
            "responseBody": {
                "application/json": {"body": {"error": error_msg}}
            }
        }
    }

def build_success_response(event, response_body):
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event.get("actionGroup"),
            "apiPath": event.get("apiPath"),
            "httpMethod": event.get("httpMethod"),
            "httpStatusCode": 200,
            "responseBody": {
                "application/json": {"body": response_body}
            }
        }
    }
'''

print("✅ `LAMBDA_SLACK_FULL_CODE` defined.")


In [ ]:
import os
import zipfile
import io
import shutil
import subprocess

def create_slack_deployment_package():
    temp_dir = "./lambda_slack_package"
    os.makedirs(temp_dir, exist_ok=True)

    global LAMBDA_SLACK_FULL_CODE

    with open(os.path.join(temp_dir, "index.py"), "w") as f:
        f.write(LAMBDA_SLACK_FULL_CODE)

    # requests 라이브러리 설치 (requests는 PyGithub에 포함되어 있을 수 있으나, 명시적으로 포함)
    subprocess.check_call(['pip', 'install', 'requests', '-t', temp_dir])

    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(temp_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, temp_dir)
                zf.write(file_path, arcname)
    zip_buffer.seek(0)

    shutil.rmtree(temp_dir)

    return zip_buffer.getvalue()

slack_deployment_package = create_slack_deployment_package()
print("✅ 'send_slack_message' Lambda deployment package created.")


### 새로운 Lambda 함수 배포: `codebuddy-slack-notifier`

Slack Webhook URL을 Colab Secrets에 `SLACK_WEBHOOK_URL` 이름으로 추가해주세요. 이 URL은 Slack에서 `Incoming Webhooks` 앱을 설치하여 생성할 수 있습니다.

In [ ]:
import boto3
import os
from google.colab import userdata # Colab Secrets에서 SLACK_WEBHOOK_URL 가져오기

slack_lambda_function_name = "codebuddy-slack-notifier"

try:
    slack_webhook_url = userdata.get('SLACK_WEBHOOK_URL')
    if not slack_webhook_url:
        raise ValueError("Colab Secrets에 'SLACK_WEBHOOK_URL'을 설정해주세요.")

    lambda_client = boto3.client('lambda')

    # Try to create, if exists, update
    lambda_client.create_function(
        FunctionName=slack_lambda_function_name,
        Runtime='python3.12',
        Role=lambda_role_arn,
        Handler='index.handler',
        Code={'ZipFile': slack_deployment_package},
        Environment={'Variables': {'SLACK_WEBHOOK_URL': slack_webhook_url}},
        Timeout=30
    )
    slack_lambda_function_arn = f"arn:aws:lambda:ap-northeast-2:{account_id}:function:{slack_lambda_function_name}"
    print(f"✅ Slack Lambda 함수 생성 완료: {slack_lambda_function_arn}")
except Exception as e:
    if "ResourceConflictException" in str(e):
        slack_lambda_function_arn = update_lambda_function(
            slack_lambda_function_name,
            lambda_role_arn,
            slack_webhook_url, # Pass the webhook URL as an environment variable
            slack_deployment_package
        )
        print(f"⚠️ 기존 Slack Lambda 함수 업데이트 완료: {slack_lambda_function_arn}")
    else:
        raise e

# Add permissions for Bedrock Agent to invoke this new Lambda function
try:
    lambda_client.add_permission(
        FunctionName=slack_lambda_function_name,
        StatementId='AllowBedrockAgentInvokeForSlack',
        Action='lambda:InvokeFunction',
        Principal='bedrock.amazonaws.com',
        SourceArn=f'arn:aws:bedrock:{os.environ["AWS_DEFAULT_REGION"]}:{account_id}:agent/{agent_id}'
    )
    print("✅ `codebuddy-slack-notifier` 함수에 Bedrock Agent 호출 권한 추가 완료")
except botocore.exceptions.ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        print("⚠️ `codebuddy-slack-notifier` 함수에 이미 Bedrock Agent 호출 권한이 추가되어 있습니다.")
    else:
        raise e


### `send_slack_message` 툴 테스트

이제 새로 배포된 Slack Lambda 함수를 직접 호출하여 메시지가 전송되는지 테스트해봅니다.

In [ ]:
lambda_client = boto3.client('lambda')

# 테스트를 위한 파라미터 설정
# 여기에 실제 Slack 채널 이름 (예: '#general') 또는 사용자 ID를 입력하세요.
slack_channel = "#code-review" # 또는 "@yourusername"
slack_message = "안녕하세요, Bedrock Agent에서 보낸 테스트 메시지입니다!"

# Lambda 함수에 전달할 이벤트 페이로드 구성
event_payload = {
    "messageVersion": "1.0",
    "actionGroup": "GitHubPRTools", # Action Group 이름은 통합 후 사용할 이름으로 지정
    "apiPath": "/slack/send",
    "httpMethod": "POST",
    "parameters": [
        {"name": "channel", "value": slack_channel},
        {"name": "message", "value": slack_message}
    ]
}

try:
    # Lambda 함수 호출
    response = lambda_client.invoke(
        FunctionName=slack_lambda_function_name,
        InvocationType='RequestResponse', # 동기 호출
        Payload=json.dumps(event_payload)
    )

    # 응답 처리
    response_payload = json.loads(response['Payload'].read().decode('utf-8'))

    print("✅ Slack Lambda 함수 호출 성공!")
    print("--- 응답 본문 ---")
    print(json.dumps(response_payload, indent=2, ensure_ascii=False))

    if response_payload.get('response', {}).get('responseBody', {}).get('application/json', {}).get('body', {}).get('success'):
        print("🎉 Slack 메시지가 성공적으로 전송되었습니다!")
    else:
        print(f"❌ Slack 메시지 전송 실패: {response_payload}")

except Exception as e:
    print(f"❌ Slack Lambda 함수 호출 실패: {e}")


OpenAPI 스키마에 send_slack_message 추가

In [ ]:
{
  "paths": {
    "/slack/send": {
      "post": {
        "operationId": "send_slack_message",
        "summary": "Send a message to Slack",
        "description": "Sends a message to a Slack channel or direct message. Use this to notify team members about completed tasks, code review results, or any alerts.",
        "parameters": [
          {
            "name": "channel",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "Slack channel name (e.g., '#code-review') or user ID"
          },
          {
            "name": "message",
            "in": "query",
            "required": True,
            "schema": {"type": "string"},
            "description": "The message text to send"
          }
        ],
        "responses": {
          "200": {"description": "Message sent successfully"}
        }
      }
    }
  }
}

## Action Group에 Slack Tool 추가
Action Group 재업데이트

In [ ]:
import boto3
import json
import time

# final_api_schema is assumed to be defined from previous cells (combined_openapi_schema and slack_openapi_schema_for_merge)

# Ensure action_group_name is defined
action_group_name = "GitHubPRTools"

# Find or create the action group ID dynamically within this cell
target_action_group_id = find_action_group_id(agent_id, action_group_name)

# If not found after retries, it indicates an issue, but the `find_action_group_id` function is designed to handle creation if needed.
# For a more robust solution, you might want to re-add the create logic here if `find_action_group_id` can only find existing ones.
# However, based on the previous trace, it tried to create and then find, so `find_action_group_id` should eventually return an ID.

if not target_action_group_id:
    raise ValueError(f"Could not determine Action Group ID for '{action_group_name}'. Please ensure the Action Group exists or can be created.")

# Action Group 업데이트
# post_comment_lambda_function_arn은 codebuddy-github-pr-comment 람다의 ARN입니다.
# 이 람다 함수는 get_github_pr, post_pr_comment, send_slack_message 세 가지 기능을 모두 처리하도록 업데이트되어야 합니다.

updated_action_group_response = update_action_group_with_id(
    agent_id,
    target_action_group_id,
    post_comment_lambda_function_arn, # 이 Lambda가 세 가지 툴 모두를 처리하도록 업데이트되어야 함
    final_api_schema,
    action_group_name="GitHubPRTools"
)
print("✅ Action Group updated with all three tools (get_github_pr, post_pr_comment, send_slack_message).")

# Prepare 재실행
# Agent Alias도 새로 생성 (또는 업데이트) - 이 부분은 bc929955 셀에 이미 포함되어 있음
prepare_agent_with_retry(agent_id)
update_alias(agent_id, alias_id)
print("✅ Agent Prepare 완료 및 Alias 업데이트 완료 - 세 가지 Tool 사용 가능")

## 복합 시나리오 테스트 (PR 리뷰 + Slack 알림)
한 번의 질문으로 여러 Tool 실행

In [ ]:
def test_multi_tool_scenario(agent_id, alias_id):
    prompt = """
    다음 작업을 수행해주세요:
    1. 내 저장소 my-org/my-repo의 PR #123 정보를 가져오고
    2. 코드 리뷰 결과를 '코드가 깔끔합니다! 수고하셨어요' 댓글로 남겨주세요
    3. 완료되면 #code-review 채널에 'PR #123 리뷰 완료' 메시지를 보내주세요
    """
    bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')
    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId="multi-tool-session",
        inputText=prompt,
        enableTrace=True
    )

    for event in response['completion']:
        if 'chunk' in event:
            print(event['chunk']['bytes'].decode('utf-8'))
        elif 'trace' in event:
            trace = event['trace']['trace']
            if 'orchestrationTrace' in trace:
                orch = trace['orchestrationTrace']
                rationale = orch.get('rationale', '')
                if rationale:
                    print(f"\n🧠 Agent 생각: {rationale[:200]}...")

## Agent 의사결정 과정 시각화
Trace 정보로 시퀀스 다이어그램 그리기

In [ ]:
def visualize_agent_flow(agent_id, alias_id, prompt):
    bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')
    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId="viz-session",
        inputText=prompt,
        enableTrace=True
    )

    steps = []
    for event in response['completion']:
        if 'trace' in event:
            trace = event['trace']['trace']
            if 'orchestrationTrace' in trace:
                orch = trace['orchestrationTrace']
                # 추론 단계 저장
                if 'rationale' in orch:
                    steps.append(f"💡 생각: {orch['rationale']}")
                # 도구 호출 정보
                if 'invocationInput' in orch:
                    inv = orch['invocationInput']
                    steps.append(f"🔧 도구 호출: {inv.get('actionGroup', '')} - {inv.get('apiPath', '')}")

    # 시퀀스 출력
    print("=== Agent 실행 순서 ===")
    for i, step in enumerate(steps, 1):
        print(f"{i}. {step}")

    return steps

# 실행
prompt = "PR #123 리뷰하고, 완료되면 Slack 알림 보내줘"
visualize_agent_flow(agent_id, alias_id, prompt)